# FVF drift over trials

Does array coverage (the fraction of a trial's 180 icons that fall within a subject's FVF - see
`array_coverage.ipynb`) shrink or grow between trial 1 and trial 60? Uses trial number as a fatigue/practice
proxy, the same way `time_on_task.ipynb` does for LWS probability.
<br><br>
To statistically test this, we fit a Generalized Additive Model (GAM) with a smooth term for trial number and
subject-level random effects, in `analysis/R/fvf_over_trials_gam.R`.

In [ ]:
import os

import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.io as pio

import constants as cnst
import config as cnfg
from analysis.helpers.read_data import load_data
from pipeline.stage2_align.fixations_to_icons import fixations_to_icons
from analysis.fvf.fvf import estimate_fvf
from analysis.helpers.plotting.gam_overlay import plot_gam_predictions_ribbon

pio.renderers.default = 'notebook'      # 'notebook' or 'browser'

### Build the per-trial coverage table

Same computation as `array_coverage.ipynb` - see that notebook for the per-step rationale.

In [ ]:
loaded_data = load_data(cnfg.OUTPUT_PATH)
fixations = loaded_data.fixations
icons = loaded_data.icons
metadata = loaded_data.metadata

fvf_by_subject = estimate_fvf(loaded_data.fixation_target_dists)['selection_hazard'].drop(index='all')

all_icon_dists = fixations_to_icons(fixations, icons, metadata)
closest = (
    all_icon_dists
    .groupby([cnst.SUBJECT_STR, cnst.TRIAL_STR, cnst.ICON_STR], observed=True)[cnst.DISTANCE_DVA_STR]
    .min()
    .reset_index()
)
closest['fvf_radius'] = closest[cnst.SUBJECT_STR].map(fvf_by_subject)
closest = closest.dropna(subset=['fvf_radius'])
closest['within_fvf'] = closest[cnst.DISTANCE_DVA_STR] <= closest['fvf_radius']

array_coverage = (
    closest
    .groupby([cnst.SUBJECT_STR, cnst.TRIAL_STR], observed=True)['within_fvf']
    .agg(n_icons='size', n_covered='sum')
    .reset_index()
)
array_coverage['coverage_pct'] = 100 * array_coverage['n_covered'] / array_coverage['n_icons']
array_coverage.head()

### Visualize (empirical)

In [ ]:
overall_stats = (
    array_coverage
    .groupby(cnst.TRIAL_STR, observed=True)
    .agg(mean_coverage=('coverage_pct', 'mean'), sem_coverage=('coverage_pct', 'sem'))
    .reset_index()
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=overall_stats[cnst.TRIAL_STR], y=overall_stats['mean_coverage'],
    error_y=dict(type='data', array=overall_stats['sem_coverage'], visible=True),
    mode='markers', name='Empirical mean coverage',
    marker=dict(color=cnfg.get_discrete_color('all'), size=6),
))
fig.update_layout(
    title='Array coverage by trial number (empirical)',
    xaxis_title='Trial number', yaxis_title='Array coverage (%)',
    template='plotly_white',
)
fig.show()

### Export for the R GAM

`fvf_over_trials_gam.R` reads this CSV by hand (mirrors `funnel_results.csv` - gitignored, not read directly
by Python).

In [ ]:
outfile = os.path.join(os.getcwd(), 'R', 'array_coverage_results.csv')
array_coverage.to_csv(outfile, index=False)
outfile

### Statistical analysis

Run from the repo root:
```bash
Rscript analysis/R/fvf_over_trials_gam.R
```
$$ \text{coverage}_{covered}/\text{coverage}_{total} \sim s(\text{trial number}) + (1|\text{subject}) $$
The smooth term $s(\text{trial number})$ captures potential non-linear drift in FVF-based coverage across the
session; the random effect $(1|\text{subject})$ accounts for variability across subjects. Modeled as a binomial
count (icons covered out of 180 per trial) rather than a continuous proportion, so trial-to-trial variance
scales with the actual denominator.
<br><br>
#### Prediction and visualization

In [ ]:
plot_gam_predictions_ribbon(
    os.path.join(os.getcwd(), 'R', 'fvf_over_trials_predictions.csv'),
    prob_col='coverage_prop',
    title='GAM-Estimated Array Coverage Over Trials',
    y_title='Predicted array coverage',
    x_tickvals=[1, 10, 20, 30, 40, 50, 60],
)